# 09. HAR model — one year (Method #1)
The first benchmark, **HAR**, is a **Linear Regression** fitted with `statsmodels` `OLS` — the same tool as the class notebook *LR (Simple Salary)*. In this notebook we build it for **one** test year: train on 2003–2006, test on 2007.

* **Needs:** `work/dataset.csv`. **Makes:** nothing new (notebook 10 does all years).

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv
dataset = read_csv(folder + 'work/dataset.csv', index_col=0, parse_dates=True)
dataset = dataset.loc[:'2023-08-31']        # main sample
print(dataset.shape)

(5243, 17)


## The idea
Next week's volatility depends on **today's**, **last week's** and **last month's** volatility (Corsi, 2009) — like traders who look at different horizons:

$$\ln RV5 = B_0 + B_d \cdot \ln(\text{today}) + B_w \cdot \ln(\text{last week}) + B_m \cdot \ln(\text{last month})$$

Compare with the Salary notebook: *Salary = B0 + B1 × YearsExperience*. Here there are three x's instead of one, and everything is in logs (notebook 04 explained why).

## Divide data to x and y

In [3]:
from numpy import log, exp
x = log(dataset[['rv_d', 'rv_w', 'rv_m']].clip(lower=0.00000001))
y = log(dataset['rv5_fwd'])
x.tail()

,rv_d,rv_w,rv_m
2023-08-25,-10.756897,-9.821343,-9.465458
2023-08-28,-8.846043,-9.436785,-9.519198
2023-08-29,-11.647985,-9.928011,-9.514191
2023-08-30,-10.757307,-10.131118,-9.507905
2023-08-31,-9.526576,-9.833354,-9.475105


## Data Splitting (by time)
* **Training:** 2003 – 2006, **without the last 5 days of 2006**: their target (the next 5 days) reaches into 2007, the test year — that would be peeking at the answers.
* **Testing:** every day of 2007.

In [4]:
train_days = dataset.loc[:'2006-12-31'].dropna(subset=['rv5_fwd', 'rv_m']).index
train_days = train_days[:-5]
test_days = dataset.loc['2007-01-01':'2007-12-31'].dropna(subset=['rv5_fwd']).index
print(len(train_days), 'training days')
print(len(test_days), 'test days')

1008 training days
254 test days


In [5]:
x_train = x.loc[train_days]
y_train = y.loc[train_days]
x_test = x.loc[test_days]
y_test = dataset.loc[test_days, 'rv5_fwd']      # the realised variance we want to forecast
print(x_train.shape)
print(x_test.shape)

(1008, 3)
(254, 3)


# Linear Regression (HAR)
## Modelling

In [6]:
from statsmodels.api import OLS, add_constant
x_train = add_constant(x_train)
model = OLS(y_train, x_train)      # building
best_model = model.fit()           # training
print(best_model.summary())

                            OLS Regression Results                            
Dep. Variable:                rv5_fwd   R-squared:                       0.113
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     42.64
Date:                Thu, 01 Oct 2026   Prob (F-statistic):           6.10e-26
Time:                        04:31:29   Log-Likelihood:                -1278.4
No. Observations:                1008   AIC:                             2565.
Df Residuals:                    1004   BIC:                             2584.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.8334      0.408     -9.384      0.0

In [7]:
best_model.params

const   -3.833383
rv_d     0.006900
rv_w     0.035312
rv_m     0.425291
dtype: float64

Read it like the Salary equation, for example:

**log RV5 = −3.83 + 0.007 × log(today) + 0.035 × log(last week) + 0.425 × log(last month)**

The **monthly** term has the biggest weight: the slow level of volatility matters most.

In [8]:
print('R2 =', round(best_model.rsquared * 100, 2), '%')

R2 = 11.3 %


The R² is only about 11%: the calm 2003–2006 years are hard to learn from (there was little to learn). Notebook 10 shows how R² grows once crisis years are in the training data.

## Testing
The model predicts the **log** of the variance. To go back we use `exp(...)`. But $\exp$ of an average log is a little too small on average, so we multiply by a **smearing factor** (Duan): the average of $\exp$(residual) in the training data.

In [9]:
x_test = add_constant(x_test, has_constant='add')
log_pred = best_model.predict(x_test)          # testing (in logs)
smear = exp(best_model.resid).mean()
print('smearing factor =', round(smear, 3))
y_pred1 = exp(log_pred) * smear
y_pred1.head()

smearing factor = 1.493


2007-01-02    0.000363
2007-01-03    0.000364
2007-01-04    0.000356
2007-01-05    0.000350
2007-01-08    0.000362
dtype: float64

## Evaluation

In [10]:
ratio = y_test / y_pred1
QLIKE1 = (ratio - log(ratio) - 1).mean()
print('QLIKE (HAR, 2007) =', round(QLIKE1, 3))

QLIKE (HAR, 2007) = 0.659


Check: are these the same forecasts as in the project's saved results?

In [11]:
saved = read_csv(folder + 'results/benchmark_forecasts.csv', index_col=0, parse_dates=True)
print('largest difference from the saved HAR forecasts:', (y_pred1 - saved.loc[test_days, 'HAR']).abs().max())

largest difference from the saved HAR forecasts: 9.969238976004213e-17


# Prediction
Like `best_model.predict([[1, 20]])` in the Salary notebook: a **new** situation. Suppose the ISEQ moved **5%** today, about **2%** a day over the last week and **1.5%** a day over the last month. What does HAR expect for next week?

In [12]:
from pandas import DataFrame
from numpy import sqrt
new1 = DataFrame([{'const': 1, 'rv_d': log(0.05 ** 2), 'rv_w': log(0.02 ** 2), 'rv_m': log(0.015 ** 2)}])
new_pred = exp(best_model.predict(new1)) * smear
print('forecast for next week (annualised volatility):', round(sqrt(new_pred[0] * 252 / 5) * 100, 1), '%')

forecast for next week (annualised volatility): 18.2 %
